# MOANA & Megafauna: PACE Phytoplankton Communities and Marine Mammal Habitat Use
### NEFSC Mid-Atlantic Marine Mammal Visual Survey × NASA PACE Satellite Data
**2026 PACE Hackathon | Liz Ferguson, Ana Vaz, James King**

This notebook integrates NOAA NEFSC cetacean survey observations with NASA PACE satellite
data products to characterize the environmental niche of marine megafauna in the
Mid-Atlantic Bight (Dec 2024 – Mar 2025). Phytoplankton community structure
(MOANA: Synechococcus, Prochlorococcus, Picoeukaryotes), chlorophyll-a, particulate
organic carbon (PACE OCI), sea surface temperature (MODIS Aqua), and bathymetry
(GEBCO) are co-located with individual sighting records.

**Notebook Sections:**
1. Setup & Authentication  
2. Load Survey Observations  
3. Satellite Data Extraction (queries EarthAccess; appends to CSV)  
4. Study Area Maps  
5. Environmental Conditions by Species  
6. Environmental Niche Analysis  
7. Multivariate Analysis: Principal Component Analysis (PCA)


*¹Ocean Science Analytics | ²[Affiliation TBC] | ³[Affiliation TBC]*

## 1. Setup & Authentication

Import required packages and authenticate with NASA EarthAccess.
Credentials are cached in `~/.netrc` after the first interactive login.

In [ ]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import seaborn as sns
import earthaccess
from tqdm import tqdm
from scipy import stats
from scipy.stats import kruskal, mannwhitneyu
from itertools import combinations
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# ── Common plot style ──────────────────────────────────────────────────────
sns.set_style("whitegrid")
plt.rcParams.update({"figure.dpi": 150, "font.size": 11})

# ── Study area bounding box ─────────────────────────────────────────────────
LON_MIN, LON_MAX = -77.0, -71.5
LAT_MIN, LAT_MAX = 35.5, 41.0
BBOX = (LON_MIN, LAT_MIN, LON_MAX, LAT_MAX)   # (west, south, east, north) for earthaccess

# ── Species colours consistent throughout ─────────────────────────────────
SPECIES_COLORS = {
    'Fin whale':                    '#e41a1c',
    'Humpback whale':               '#377eb8',
    'Common minke whale':           '#4daf4a',
    'Sperm whale':                  '#984ea3',
    'Beaked whales':                '#ff7f00',
    'North Atlantic right whale':   '#a65628',
    'Blue whale':                   '#f781bf',
}

# ── Data paths ─────────────────────────────────────────────────────────────
os.makedirs("data", exist_ok=True)
os.makedirs("figures", exist_ok=True)

print("Packages loaded.")

In [ ]:
# ── Figure export settings (RSE requires TIFF, ≥300 dpi) ─────────────────
# All figures are saved as TIFF at 300 dpi to meet Remote Sensing of
# Environment submission requirements (vector EPS preferred for line art,
# but TIFF 300 dpi is the universal halftone standard).

import os

FIGURE_DIR = "figures"
os.makedirs(FIGURE_DIR, exist_ok=True)

def save_fig(fname, dpi=300):
    """Save current figure as TIFF (and optionally PDF for vector backup)."""
    tiff_path = os.path.join(FIGURE_DIR, fname)
    plt.savefig(tiff_path, dpi=dpi, bbox_inches='tight',
                format='tiff', pil_kwargs={'compression': 'tiff_lzw'})
    print(f"  Saved: {tiff_path}")

print("Figure export helper defined. Output directory:", FIGURE_DIR)

In [ ]:
# Authenticate with NASA EarthAccess (saves credentials to ~/.netrc)
auth = earthaccess.login()
if not auth.authenticated:
    auth.login(strategy="interactive", persist=True)
print("Authenticated:", auth.authenticated)

## 2. Load Survey Observations

The NEFSC Mid-Atlantic marine mammal visual survey (Dec 2024 – Mar 2025) is archived
on OBIS-SEAMAP (dataset 2368, DOI 10.82144/4431190b). The CSV exported from SEAMAP
contains one row per sighting with latitude, longitude, UTC datetime, species, and
group size. Point this path to your local copy.

In [ ]:
# ── Load raw survey observations ─────────────────────────────────────────
OBS_RAW = "data/OBIS_NEFSC_offshore_obs.csv"   # ← update path if needed

obs = pd.read_csv(OBS_RAW)
obs['date'] = pd.to_datetime(obs['date'])
obs['date_only'] = obs['date'].dt.date

print(f"Loaded {len(obs):,} observations from {obs['date'].min().date()} to {obs['date'].max().date()}")
print("\nSpecies counts:")
print(obs['common_name'].value_counts())
print(f"\nUnique survey dates: {obs['date_only'].nunique()}")

## 3. Satellite Data Extraction

Each subsection below queries a different satellite product via `earthaccess`,
extracts values within a 3×3 pixel window centred on each observation, and
appends summary statistics to a running dataframe. At the end all products are
merged and saved as `data/NEFSC_All_Variables.csv`.

> **Note:** If you already have `data/NEFSC_All_Variables.csv` you can skip to
> Section 4. Re-running these cells overwrites the file.

**Products used:**
| Variable | Product | Satellite / Sensor | Resolution |
|---|---|---|---|
| Chlorophyll-a | `PACE_OCI_L3M_CHL` | PACE / OCI | ~1 km |
| Phytoplankton carbon | `PACE_OCI_L3M_CARBON` | PACE / OCI | ~1 km |
| Synechococcus / Prochlorococcus / Picoeukaryotes | `PACE_OCI_L4M_MOANA` | PACE / OCI | 4 km |
| Sea surface temperature | `MODIS_AQUA_L3_SST_THERMAL_DAILY_4KM_DAYTIME_V2019.0` | Aqua / MODIS | 4 km |
| Bathymetry (depth) | GEBCO 2025 `.nc` file | N/A (static grid) | ~450 m |
| SSH anomaly | AVISO / CMEMS – see note | altimetry | 0.25° |
| Optical water type (OWT) classification | `PACE_OCI_L3M_AOP` (hyperspectral Rrs) | PACE / OCI | 4 km |


### 3a. Helper: pixel-window extraction

A shared function extracts the mean, median, std, min, max, Q25, and Q75 of all
valid (non-NaN) pixels within a ±1 pixel window around each observation point.

In [ ]:
def extract_window_stats(da, lat_vals, lon_vals, lat_c, lon_c, half=1):
    """Extract summary stats from a ±half-pixel window centred on (lat_c, lon_c)."""
    lat_idx = int(np.abs(lat_vals - lat_c).argmin())
    lon_idx = int(np.abs(lon_vals - lon_c).argmin())
    lat_sl = slice(max(lat_idx - half, 0), min(lat_idx + half + 1, len(lat_vals)))
    lon_sl = slice(max(lon_idx - half, 0), min(lon_idx + half + 1, len(lon_vals)))
    window = da.isel(lat=lat_sl, lon=lon_sl).values.flatten()
    window = window[~np.isnan(window)]
    if len(window) == 0:
        return {k: np.nan for k in ['mean','median','std','min','max','q25','q75']}
    return {
        'mean':   float(np.mean(window)),
        'median': float(np.median(window)),
        'std':    float(np.std(window)),
        'min':    float(np.min(window)),
        'max':    float(np.max(window)),
        'q25':    float(np.percentile(window, 25)),
        'q75':    float(np.percentile(window, 75)),
    }

print("Helper function defined.")

### 3b. PACE OCI Chlorophyll-a

We use 8-day composites (`*.8D.*.9km.*` granules) of the PACE OCI Level-3 mapped
chlorophyll-a product. An 8-day window centred on each observation date is used
to maximise cloud-free coverage.

In [ ]:
# ── Query and extract PACE CHL ─────────────────────────────────────────────
df_chl = obs[['latitude','longitude','date','date_only','common_name',
               'scientific_name','group_size']].copy()

unique_dates = sorted(df_chl['date_only'].unique())
chl_records  = {idx: {} for idx in df_chl.index}

for date in tqdm(unique_dates, desc="Extracting CHL"):
    d      = pd.Timestamp(date)
    start  = (d - pd.Timedelta(days=4)).strftime("%Y-%m-%d")
    end    = (d + pd.Timedelta(days=4)).strftime("%Y-%m-%d")
    mask   = df_chl['date_only'] == date

    try:
        results = earthaccess.search_data(
            short_name  = 'PACE_OCI_L3M_CHL',
            temporal    = (start, end),
            granule_name= "*.8D.*.9km.*",
            bounding_box= BBOX
        )
        if not results:
            continue
        fileset = earthaccess.open([results[0]])
        with xr.open_dataset(fileset[0], engine="h5netcdf") as ds:
            chl_da   = ds['chlor_a'].squeeze()
            lat_vals = ds.lat.values
            lon_vals = ds.lon.values
        for idx, row in df_chl[mask].iterrows():
            s = extract_window_stats(chl_da, lat_vals, lon_vals,
                                     row['latitude'], row['longitude'])
            chl_records[idx] = {f'chlor_a_{k}': v for k, v in s.items()}
    except Exception as e:
        print(f"  CHL error on {date}: {e}")

# Merge CHL columns into dataframe
chl_df = pd.DataFrame.from_dict(chl_records, orient='index')
df_chl = df_chl.join(chl_df)
print(f"CHL extraction complete. Non-null rows: {df_chl['chlor_a_median'].notna().sum()}")

### 3c. PACE OCI Phytoplankton Carbon

Particulate organic carbon (phytoplankton carbon) from the PACE OCI Level-3 mapped
Carbon product. Same 8-day composite approach as for CHL.

In [ ]:
# ── Query and extract PACE Carbon ─────────────────────────────────────────
carbon_records = {idx: {} for idx in df_chl.index}

for date in tqdm(unique_dates, desc="Extracting Carbon"):
    d     = pd.Timestamp(date)
    start = (d - pd.Timedelta(days=4)).strftime("%Y-%m-%d")
    end   = (d + pd.Timedelta(days=4)).strftime("%Y-%m-%d")
    mask  = df_chl['date_only'] == date

    try:
        results = earthaccess.search_data(
            short_name  = 'PACE_OCI_L3M_CARBON',
            temporal    = (start, end),
            granule_name= "*.8D.*.9km.*",
            bounding_box= BBOX
        )
        if not results:
            continue
        fileset = earthaccess.open([results[0]])
        with xr.open_dataset(fileset[0], engine="h5netcdf") as ds:
            # Variable name may be 'carbon_phyto' or 'phyto_carbon' – check ds.data_vars
            var_name = [v for v in ds.data_vars if 'carbon' in v.lower() or 'phyto' in v.lower()][0]
            carb_da  = ds[var_name].squeeze()
            lat_vals = ds.lat.values
            lon_vals = ds.lon.values
        for idx, row in df_chl[mask].iterrows():
            s = extract_window_stats(carb_da, lat_vals, lon_vals,
                                     row['latitude'], row['longitude'])
            carbon_records[idx] = {f'carbon_phyto_{k}': v for k, v in s.items()}
    except Exception as e:
        print(f"  Carbon error on {date}: {e}")

carb_df = pd.DataFrame.from_dict(carbon_records, orient='index')
df_chl  = df_chl.join(carb_df)
print(f"Carbon extraction complete. Non-null rows: {df_chl['carbon_phyto_median'].notna().sum()}")

### 3d. PACE MOANA – Phytoplankton Community Composition

The PACE OCI Level-4 MOANA product provides modelled concentrations of three
phytoplankton size classes: Synechococcus, Prochlorococcus, and Picoeukaryotes
(cells mL⁻¹). We use 8-day, 4 km granules.

In [ ]:
# ── Query and extract PACE MOANA ─────────────────────────────────────────
moana_records = {idx: {} for idx in df_chl.index}

MOANA_VARS = {
    'syncoccus_moana':  'synechococcus',
    'prococcus_moana':  'prochlorococcus',
    'picoeuk_moana':    'picoeukaryotes',
}

for date in tqdm(unique_dates, desc="Extracting MOANA"):
    d     = pd.Timestamp(date)
    start = (d - pd.Timedelta(days=7)).strftime("%Y-%m-%d")
    end   = (d + pd.Timedelta(days=7)).strftime("%Y-%m-%d")
    mask  = df_chl['date_only'] == date

    try:
        results = earthaccess.search_data(
            short_name  = 'PACE_OCI_L4M_MOANA',
            temporal    = (start, end),
            granule_name= "*.8D.*.4km.*",
            bounding_box= BBOX
        )
        if not results:
            print(f"  No MOANA data for {date}")
            continue
        fileset = earthaccess.open([results[0]])
        with xr.open_dataset(fileset[0], engine="h5netcdf") as ds:
            lat_vals = ds.lat.values
            lon_vals = ds.lon.values
            arrays   = {v: ds[v].squeeze() for v in MOANA_VARS if v in ds}

        for idx, row in df_chl[mask].iterrows():
            row_rec = {}
            for nc_var, col_prefix in MOANA_VARS.items():
                if nc_var in arrays:
                    s = extract_window_stats(arrays[nc_var], lat_vals, lon_vals,
                                             row['latitude'], row['longitude'])
                    row_rec.update({f'{col_prefix}_{k}': v for k, v in s.items()})
            moana_records[idx] = row_rec
    except Exception as e:
        print(f"  MOANA error on {date}: {e}")

moana_df = pd.DataFrame.from_dict(moana_records, orient='index')
df_chl   = df_chl.join(moana_df)
print(f"MOANA extraction complete. Synechococcus non-null: {df_chl['synechococcus_median'].notna().sum()}")

### 3e. Optical Water Type (OWT) Classification & MOANA Quality Masking

MOANA's picophytoplankton retrievals are tuned for open-ocean (case 1) waters, so
their reliability is expected to degrade in optically complex, turbid coastal
waters. Following the [Fish-PACE Hackweek 2025 water classification
tutorial](https://fish-pace.github.io/hackweek-2025/presentations/notebooks/water_classification.html)
(Wei et al. 2022), we classify each PACE pixel into one of 23 reference optical
water types (OWTs) using a cosine-distance match between hyperspectral remote
sensing reflectance (R<sub>rs</sub>) and a reference spectral library. OWT
classes 1–15 correspond to clearer, blue-water optical types, while classes
16–23 correspond to increasingly turbid / coastal water types.

For every date with observations, we classify the water type on the same grid
used for the MOANA extraction, attach the water class to each observation, and
then **mask (set to NaN) the MOANA cell values for any observation in a turbid
water class (16–23)**, since MOANA is not considered reliable there. The
`water_class` column is retained in the combined dataset either way, so it can
still be used downstream (e.g. as a covariate, or to check how many nearshore
observations were excluded from MOANA-based analyses).

> **Note:** This section defines `classify_water_type()` and
> `TURBID_OWT_CLASSES`, which are also used later when mapping MOANA (Section
> 4c) to shade turbid pixels gray. If skipping ahead to Section 4, re-run this
> section's cells (in addition to Section 3a) so those helpers are defined.


In [ ]:
# ── Optical Water Type (OWT) classification helpers ───────────────────────
# Following Wei et al. (2022) / Fish-PACE Hackweek 2025:
# https://fish-pace.github.io/hackweek-2025/presentations/notebooks/water_classification.html
# 23 reference optical water classes, ordered from clear/blue (1) to
# turbid/coastal (23). MOANA is unreliable in turbid classes, so we exclude
# classes 16-23 from MOANA-based analysis.

OWT_REF_URL = ("https://raw.githubusercontent.com/fish-pace/2025-tutorials/main/"
               "Supporting_files/Hyperspectral_nRrs.csv")
TURBID_OWT_CLASSES = list(range(16, 24))   # OWT classes 16-23 = turbid/coastal

_owt_ref_cache = None

def load_owt_reference():
    """Load and cache the 23-class reference nRrs spectra (Wei et al. 2022)."""
    global _owt_ref_cache
    if _owt_ref_cache is None:
        _owt_ref_cache = pd.read_csv(OWT_REF_URL)
    return _owt_ref_cache


def classify_water_type(rrs_da):
    """Classify each pixel of a hyperspectral Rrs DataArray (wavelength, lat, lon)
    into one of 23 optical water types via cosine distance to the Wei et al.
    (2022) reference spectra. Returns an xarray.DataArray of water class
    (1-23; NaN where no valid spectrum), or None if wavelengths don't overlap."""
    nRrs_ref = load_owt_reference()
    wavelengths = rrs_da['wavelength'].values

    sat_wavelengths_str = [str(int(w)) for w in wavelengths]
    ref_columns = nRrs_ref.columns
    common_wavelengths = sorted(set(sat_wavelengths_str).intersection(ref_columns), key=int)
    if not common_wavelengths:
        return None

    ref = nRrs_ref[common_wavelengths].values
    common_wavelengths_float = [float(w) for w in common_wavelengths]
    rrs_common = rrs_da.sel(wavelength=common_wavelengths_float, method="nearest")

    # Normalize satellite spectra (unit vector per pixel)
    norm = np.sqrt((rrs_common ** 2).sum(dim='wavelength'))
    rrs_norm = rrs_common / norm

    # Normalize reference spectra
    ref_norm = ref / np.linalg.norm(ref, axis=1, keepdims=True)

    # Flatten spatial dims and compute cosine distance to all 23 references
    rrs_flat = rrs_norm.stack(pix=('lat', 'lon')).transpose('pix', 'wavelength').values
    dot_products = np.dot(rrs_flat, ref_norm.T)
    cosine_distances = 1 - dot_products

    class_ids = np.full((cosine_distances.shape[0],), np.nan)
    valid = np.where(~np.isnan(cosine_distances).all(axis=1))[0]
    class_ids[valid] = np.argmin(cosine_distances[valid], axis=1) + 1  # 1-based

    lat_dim, lon_dim = rrs_norm.sizes['lat'], rrs_norm.sizes['lon']
    class_map = xr.DataArray(
        class_ids.reshape(lat_dim, lon_dim), dims=('lat', 'lon'),
        coords={'lat': rrs_norm['lat'], 'lon': rrs_norm['lon']}, name='water_class')
    return class_map

print("OWT classification helpers defined. Turbid classes:", TURBID_OWT_CLASSES)


In [ ]:
# ── Query and classify water type for each survey date ────────────────────
owt_records    = {idx: {} for idx in df_chl.index}
owt_class_maps = {}   # cache per-date class_map, reused for mapping in Section 4c

for date in tqdm(unique_dates, desc="Classifying OWT"):
    d     = pd.Timestamp(date)
    start = (d - pd.Timedelta(days=7)).strftime("%Y-%m-%d")
    end   = (d + pd.Timedelta(days=7)).strftime("%Y-%m-%d")
    mask  = df_chl['date_only'] == date

    try:
        results = earthaccess.search_data(
            short_name  = 'PACE_OCI_L3M_AOP',
            temporal    = (start, end),
            granule_name= "*.8D.*.4km.*",
            bounding_box= BBOX
        )
        if not results:
            print(f"  No hyperspectral Rrs (AOP) data for OWT on {date}")
            continue

        fileset = earthaccess.open([results[0]])
        with xr.open_dataset(fileset[0], engine="h5netcdf") as ds:
            ds_sub = ds.sel(lat=slice(LAT_MAX, LAT_MIN), lon=slice(LON_MIN, LON_MAX))
            class_map = classify_water_type(ds_sub['Rrs'])

        if class_map is None:
            print(f"  Could not classify OWT on {date} (no overlapping wavelengths)")
            continue
        owt_class_maps[date] = class_map

        lat_vals = class_map.lat.values
        lon_vals = class_map.lon.values

        for idx, row in df_chl[mask].iterrows():
            # Modal (most common) water class within the same ±1 pixel window
            # used for the other satellite extractions
            lat_idx = int(np.abs(lat_vals - row['latitude']).argmin())
            lon_idx = int(np.abs(lon_vals - row['longitude']).argmin())
            lat_sl  = slice(max(lat_idx - 1, 0), min(lat_idx + 2, len(lat_vals)))
            lon_sl  = slice(max(lon_idx - 1, 0), min(lon_idx + 2, len(lon_vals)))
            window_vals = class_map.isel(lat=lat_sl, lon=lon_sl).values.flatten()
            window_vals = window_vals[~np.isnan(window_vals)]
            if len(window_vals) == 0:
                owt_records[idx] = {'water_class': np.nan}
                continue
            vals, counts = np.unique(window_vals, return_counts=True)
            owt_records[idx] = {'water_class': float(vals[np.argmax(counts)])}
    except Exception as e:
        print(f"  OWT error on {date}: {e}")

owt_df = pd.DataFrame.from_dict(owt_records, orient='index')
df_chl = df_chl.join(owt_df)
print(f"OWT classification complete. Non-null water_class: {df_chl['water_class'].notna().sum()}")
print("\nWater class distribution:")
print(df_chl['water_class'].value_counts().sort_index())


In [ ]:
# ── Mask MOANA cell values in turbid water classes (16-23) ────────────────
# MOANA is tuned for open-ocean (case 1) waters; classes 16-23 correspond to
# increasingly turbid/coastal optical water types (Wei et al. 2022) where
# MOANA retrievals are not considered reliable. We set the MOANA summary
# statistics to NaN for any observation whose (modal) water class falls in
# this turbid range. `water_class` itself is kept so the exclusion is
# auditable and can be used downstream.
moana_stat_cols = [c for c in df_chl.columns
                    if c.startswith(('synechococcus_', 'prochlorococcus_', 'picoeukaryotes_'))]

turbid_obs_mask = df_chl['water_class'].isin(TURBID_OWT_CLASSES)
n_masked = int(turbid_obs_mask.sum())

df_chl.loc[turbid_obs_mask, moana_stat_cols] = np.nan

print(f"Masked MOANA values for {n_masked:,} of {len(df_chl):,} observations "
      f"in turbid water classes (16-23).")
print(f"Remaining valid MOANA (Synechococcus) observations: "
      f"{df_chl['synechococcus_median'].notna().sum():,}")


### 3f. MODIS Aqua Sea Surface Temperature

SST comes from MODIS Aqua rather than PACE, so its spatial resolution (~4 km)
is lower than the OCI products. We search for daily L3 mapped daytime SST.
The `short_name` below works as of early 2026; verify at
https://search.earthdata.nasa.gov if granules are not found.

In [ ]:
# ── Query and extract MODIS SST ──────────────────────────────────────────
# Short name for MODIS Aqua L3 SST 4 km daily (daytime)
# Verify at https://search.earthdata.nasa.gov if needed
MODIS_SST_SHORTNAME = "MODIS_AQUA_L3_SST_THERMAL_DAILY_4KM_DAYTIME_V2019.0"

sst_records = {idx: {} for idx in df_chl.index}

for date in tqdm(unique_dates, desc="Extracting SST"):
    d     = pd.Timestamp(date)
    start = (d - pd.Timedelta(days=4)).strftime("%Y-%m-%d")
    end   = (d + pd.Timedelta(days=4)).strftime("%Y-%m-%d")
    mask  = df_chl['date_only'] == date

    try:
        results = earthaccess.search_data(
            short_name  = MODIS_SST_SHORTNAME,
            temporal    = (start, end),
            bounding_box= BBOX
        )
        if not results:
            print(f"  No SST data for {date}")
            continue
        fileset = earthaccess.open([results[0]])
        with xr.open_dataset(fileset[0], engine="h5netcdf") as ds:
            # Variable is typically 'sst' or 'sea_surface_temperature'
            sst_var  = [v for v in ds.data_vars if 'sst' in v.lower()][0]
            sst_da   = ds[sst_var].squeeze()
            lat_vals = ds.lat.values
            lon_vals = ds.lon.values

        for idx, row in df_chl[mask].iterrows():
            s = extract_window_stats(sst_da, lat_vals, lon_vals,
                                     row['latitude'], row['longitude'])
            sst_records[idx] = {f'sst_{k}': v for k, v in s.items()}
    except Exception as e:
        print(f"  SST error on {date}: {e}")

sst_df = pd.DataFrame.from_dict(sst_records, orient='index')
df_chl = df_chl.join(sst_df)
print(f"SST extraction complete. Non-null rows: {df_chl['sst_median'].notna().sum()}")
print("Note: SST resolution (4 km) is coarser than PACE OCI products (~1 km).")

### 3g. GEBCO Bathymetry (Depth)

Depth is extracted from a GEBCO 2025 netCDF file. Download the file cropped to the
study area from https://download.gebco.net and save it to `data/gebco_study_area.nc`.
GEBCO elevation is negative below sea level; we convert to positive depth (m).

In [ ]:
# ── Load GEBCO and extract depth at each observation ─────────────────────
GEBCO_FILE = "data/gebco_study_area.nc"   # ← update filename if needed

bathy = xr.open_dataset(GEBCO_FILE)
depth_var = 'elevation' if 'elevation' in bathy.variables else list(bathy.data_vars)[0]
print(f"Using bathymetry variable: '{depth_var}'")
print(f"Elevation range: {bathy[depth_var].min().values:.1f} to {bathy[depth_var].max().values:.1f} m")

bathy_lat = bathy.lat.values
bathy_lon = bathy.lon.values

depth_vals = []
for _, row in tqdm(df_chl.iterrows(), total=len(df_chl), desc="Extracting depth"):
    try:
        elev = float(bathy[depth_var].sel(lat=row['latitude'], lon=row['longitude'],
                                          method='nearest').values)
        depth_vals.append(-elev if elev <= 0 else 0.0)
    except Exception:
        depth_vals.append(np.nan)

df_chl['depth_m'] = depth_vals
print(f"Depth extraction complete. Non-null: {df_chl['depth_m'].notna().sum()}")
print(f"Depth range: {df_chl['depth_m'].min():.0f} – {df_chl['depth_m'].max():.0f} m")

### 3h. Sea Surface Height Anomaly (SSHA)

SSHA is not distributed through NASA EarthAccess. We used daily merged altimetry
from **AVISO / CMEMS** (SEALEVEL_GLO_PHY_L4_NRT product, 0.25° resolution).
To obtain it:
1. Register at https://marine.copernicus.eu
2. Download the daily gridded SSHA field for the study period
3. Run the extraction block below, pointing `SSHA_DIR` to your downloaded NetCDF files.

If SSHA is already included in your CSV (column `ssha`) you can skip this block.

In [ ]:
# ── Optional: extract SSHA from CMEMS NetCDF files ───────────────────────
# SSHA_DIR = "data/ssha/"   # folder containing daily CMEMS SSHA .nc files

# Uncomment and adapt if re-extracting from scratch:
# ssha_vals = []
# for _, row in tqdm(df_chl.iterrows(), total=len(df_chl), desc="Extracting SSHA"):
#     date_str = pd.Timestamp(row['date']).strftime("%Y%m%d")
#     nc_file  = os.path.join(SSHA_DIR, f"dt_global_allsat_phy_l4_{date_str}_*.nc")
#     try:
#         import glob
#         f = glob.glob(nc_file)[0]
#         with xr.open_dataset(f) as ds:
#             val = float(ds['sla'].sel(latitude=row['latitude'],
#                                       longitude=row['longitude'],
#                                       method='nearest').values)
#         ssha_vals.append(val)
#     except Exception:
#         ssha_vals.append(np.nan)
# df_chl['ssha'] = ssha_vals

# If SSHA is already in your combined CSV, load it here:
if 'ssha' not in df_chl.columns:
    print("SSHA column not found – add it from CMEMS download or skip.")
else:
    print(f"SSHA already present. Non-null: {df_chl['ssha'].notna().sum()}")

### 3i. Save Combined Dataset

All extracted variables are merged and saved. This CSV is used for all
subsequent analysis sections.

In [ ]:
# ── Save combined CSV ────────────────────────────────────────────────────
OUT_CSV = "data/NEFSC_All_Variables.csv"
df_chl.to_csv(OUT_CSV, index=False)
print(f"Saved {len(df_chl):,} rows × {len(df_chl.columns)} columns → {OUT_CSV}")
print("\nColumns:", df_chl.columns.tolist())

## 4. Study Area Maps

These maps correspond to Slides 5–8 in the presentation. We visualise the spatial
distribution of observations against bathymetry and the MOANA phytoplankton
community fields for two representative survey dates.

### 4a. Load Final Dataset (if skipping Section 3)

> If you skipped Section 3, also re-run the Section 3e cells that define `classify_water_type()` and `TURBID_OWT_CLASSES` (no need to re-run the OWT extraction loop) — the MOANA maps below use them to shade turbid-water pixels gray.


In [ ]:
# Load the merged dataset (run this cell even if you completed Section 3)
df = pd.read_csv("data/NEFSC_All_Variables.csv")
df['date'] = pd.to_datetime(df['date'])
print(f"Loaded {len(df):,} rows. Species:")
print(df['common_name'].value_counts())

### 4b. Bathymetry Map with Observations (Slide 5)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9), subplot_kw={'projection': ccrs.PlateCarree()})

# Bathymetry background
bathy_plot = bathy[depth_var].plot(
    ax=ax, cmap='Blues_r', robust=True, alpha=0.8, add_colorbar=False,
    transform=ccrs.PlateCarree()
)
plt.colorbar(bathy_plot, ax=ax, shrink=0.6, pad=0.02,
             label='Depth (m)')

# Contour lines
try:
    ax.contour(bathy.lon, bathy.lat, bathy[depth_var],
               levels=[-200, -1000, -2000], colors='white',
               linewidths=0.8, linestyles='--', transform=ccrs.PlateCarree())
except Exception:
    pass

# Observations
for sp, grp in df.drop_duplicates(subset=['latitude','longitude','date','common_name']).groupby('common_name'):
    col = SPECIES_COLORS.get(sp, 'black')
    ax.scatter(grp['longitude'], grp['latitude'], c=col, s=30, alpha=0.8,
               edgecolors='white', linewidths=0.4, label=f"{sp} (n={len(grp)})",
               transform=ccrs.PlateCarree(), zorder=5)

ax.set_extent([LON_MIN, LON_MAX, LAT_MIN, LAT_MAX], crs=ccrs.PlateCarree())
ax.coastlines(resolution='10m', linewidth=0.8)
ax.add_feature(cfeature.LAND, facecolor='lightgray', alpha=0.6)
ax.add_feature(cfeature.STATES, linewidth=0.3, edgecolor='gray')
gl = ax.gridlines(draw_labels=True, linewidth=0.4, alpha=0.5, linestyle='--')
gl.top_labels = gl.right_labels = False
ax.set_title('Marine Mammal Observations – Mid-Atlantic Bight\n(GEBCO Bathymetry)',
             fontsize=13, fontweight='bold')
ax.legend(loc='lower left', fontsize=9, framealpha=0.9, markerscale=1.5)

plt.tight_layout()
save_fig('Fig1_StudyArea_Bathymetry_Observations.tiff')
plt.show()

### 4c. MOANA Phytoplankton Community Maps (Slides 7 & 8)

We plot the three MOANA phytoplankton fields for two dates that bracket the survey
effort: 14 January 2025 and 22 February 2025. Observations within ±7 days of each
date are overlaid.

Pixels falling in turbid optical water types (OWT classes 16–23, Section 3e) are
shaded **gray** on all four panels below, since MOANA is not considered reliable
in these waters. The bottom panel combines all three phytoplankton groups into a
single false-color RGB composite (R = Synechococcus, G = Picoeukaryotes,
B = Prochlorococcus), following the [NASA MOANA visualization
tutorial](https://nasa.github.io/oceandata-notebooks/notebooks/oci/oci_moana.html);
blended colors indicate where multiple groups contribute substantially.


In [ ]:
def robust_normalize(arr, vmin=None, vmax=None):
    """Percentile-clip and rescale an array to [0, 1] for RGB display."""
    if vmin is None or vmax is None:
        vmin, vmax = np.nanpercentile(arr, [2, 98])
    return np.clip((arr - vmin) / (vmax - vmin), 0, 1)


def plot_moana_date(date_str, obs_df, save_name):
    """Plot Prochlorococcus, Synechococcus, and Picoeukaryote maps (top row) plus
    an RGB phytoplankton-community composite (bottom row) for a given date, with
    overlaid marine mammal observations. Pixels in turbid optical water types
    (OWT classes 16-23, Section 3e) are masked from MOANA and shaded gray,
    since MOANA is not considered reliable in these water types."""
    d      = pd.Timestamp(date_str)
    start  = (d - pd.Timedelta(days=7)).strftime("%Y-%m-%d")
    end    = (d + pd.Timedelta(days=7)).strftime("%Y-%m-%d")

    results = earthaccess.search_data(
        short_name='PACE_OCI_L4M_MOANA', temporal=(start, end),
        granule_name="*.8D.*.4km.*", bounding_box=BBOX)
    if not results:
        print(f"No MOANA data found around {date_str}"); return

    fileset = earthaccess.open([results[0]])
    ds      = xr.open_dataset(fileset[0], engine="h5netcdf")

    phyto_info = {
        'Prochlorococcus': {'var': 'prococcus_moana', 'cmap': plt.cm.Blues},
        'Synechococcus':   {'var': 'syncoccus_moana',  'cmap': plt.cm.Reds},
        'Picoeukaryotes':  {'var': 'picoeuk_moana',    'cmap': plt.cm.Greens},
    }

    # Subset to study area
    ds_sub = ds.sel(lat=slice(LAT_MAX, LAT_MIN), lon=slice(LON_MIN, LON_MAX))

    # ── OWT classification for this date, resampled onto the MOANA grid, to
    # mask turbid pixels (classes 16-23) out of the MOANA display ───────────
    turbid_mask_2d = None
    try:
        owt_results = earthaccess.search_data(
            short_name='PACE_OCI_L3M_AOP', temporal=(start, end),
            granule_name="*.8D.*.4km.*", bounding_box=BBOX)
        if owt_results:
            owt_fileset = earthaccess.open([owt_results[0]])
            with xr.open_dataset(owt_fileset[0], engine="h5netcdf") as owt_ds:
                owt_sub   = owt_ds.sel(lat=slice(LAT_MAX, LAT_MIN), lon=slice(LON_MIN, LON_MAX))
                class_map = classify_water_type(owt_sub['Rrs'])
            if class_map is not None:
                class_map_on_moana = class_map.interp(lat=ds_sub.lat, lon=ds_sub.lon,
                                                       method='nearest')
                turbid_mask_2d = class_map_on_moana.isin(TURBID_OWT_CLASSES)
    except Exception as e:
        print(f"  OWT classification unavailable for {date_str} map: {e}")

    # Observations within ±7 days
    nearby = obs_df[(obs_df['date'] >= d - pd.Timedelta(days=7)) &
                    (obs_df['date'] <= d + pd.Timedelta(days=7))]

    fig = plt.figure(figsize=(20, 15))
    gs  = fig.add_gridspec(2, 3, height_ratios=[1, 1.15], hspace=0.35)
    fig.suptitle(f'Phytoplankton Community Composition with Marine Mammal Observations\n'
                 f'Mid-Atlantic Bight Study Area - {date_str}',
                 fontsize=14, fontweight='bold')

    legend_handles = [mpatches.Patch(color=v, label=k)
                      for k, v in SPECIES_COLORS.items() if k in nearby['common_name'].values]

    species_markers = {'Fin whale': 'o', 'Humpback whale': 's',
                        'Common minke whale': '^', 'Sperm whale': 'D',
                        'Beaked whales': '*'}

    masked_arrays = {}
    for j, (title, info) in enumerate(phyto_info.items()):
        ax = fig.add_subplot(gs[0, j], projection=ccrs.PlateCarree())
        if info['var'] not in ds_sub:
            ax.set_title(f"{title} (not found)"); continue
        da = ds_sub[info['var']].squeeze()

        # Mask turbid-water pixels; render them gray via the colormap's 'bad' color
        if turbid_mask_2d is not None:
            da = da.where(~turbid_mask_2d)
        masked_arrays[info['var']] = da

        cmap = info['cmap'].copy()
        cmap.set_bad(color='lightgray')

        ax.set_extent([LON_MIN, LON_MAX, LAT_MIN, LAT_MAX], crs=ccrs.PlateCarree())
        ax.coastlines(resolution='10m', linewidth=0.8)
        ax.add_feature(cfeature.LAND, facecolor='dimgray', alpha=0.6)
        ax.add_feature(cfeature.STATES, linewidth=0.3, edgecolor='gray')
        gl = ax.gridlines(draw_labels=True, linewidth=0.4, alpha=0.5, linestyle='--')
        gl.top_labels = gl.right_labels = False

        img = da.plot(ax=ax, cmap=cmap, robust=True,
                      add_colorbar=False, transform=ccrs.PlateCarree())
        cbar = plt.colorbar(img, ax=ax, orientation='horizontal', pad=0.04, shrink=0.9)
        cbar.set_label(f"{title} conc. (cells mL\u207b\u00b9)", fontsize=10)

        for sp, grp in nearby.drop_duplicates(['latitude','longitude','common_name']).groupby('common_name'):
            ax.scatter(grp['longitude'], grp['latitude'],
                       c=SPECIES_COLORS.get(sp, 'black'), s=40, alpha=0.9,
                       edgecolors='white', linewidths=0.5,
                       marker=species_markers.get(sp, 'o'),
                       transform=ccrs.PlateCarree(), zorder=5)
        ax.set_title(title, fontsize=13, fontweight='bold')

        if turbid_mask_2d is not None:
            ax.text(0.01, 0.01, 'Gray = turbid OWT (16-23), MOANA masked',
                    transform=ax.transAxes, fontsize=7, color='dimgray',
                    va='bottom', ha='left',
                    bbox=dict(facecolor='white', alpha=0.75, pad=1.5, edgecolor='none'))

    # ── RGB phytoplankton-community composite (bottom row) ────────────────
    # R = Synechococcus, G = Picoeukaryotes, B = Prochlorococcus, following
    # https://nasa.github.io/oceandata-notebooks/notebooks/oci/oci_moana.html
    ax_rgb = fig.add_subplot(gs[1, :], projection=ccrs.PlateCarree())

    if all(v in masked_arrays for v in ['syncoccus_moana', 'picoeuk_moana', 'prococcus_moana']):
        syn_norm  = robust_normalize(masked_arrays['syncoccus_moana'].values)
        pico_norm = robust_normalize(masked_arrays['picoeuk_moana'].values)
        pro_norm  = robust_normalize(masked_arrays['prococcus_moana'].values)

        rgb_image = np.stack([syn_norm, pico_norm, pro_norm], axis=-1)

        # Render turbid / no-data pixels as gray rather than black
        nodata = np.isnan(syn_norm) | np.isnan(pico_norm) | np.isnan(pro_norm)
        rgb_image = np.where(nodata[..., None], 0.827, rgb_image)  # 0.827 ~ lightgray

        ax_rgb.set_extent([LON_MIN, LON_MAX, LAT_MIN, LAT_MAX], crs=ccrs.PlateCarree())
        ax_rgb.imshow(rgb_image, origin='upper',
                     extent=[float(ds_sub.lon.min()), float(ds_sub.lon.max()),
                             float(ds_sub.lat.min()), float(ds_sub.lat.max())],
                     transform=ccrs.PlateCarree())
        ax_rgb.coastlines(resolution='10m', linewidth=0.8)
        ax_rgb.add_feature(cfeature.LAND, facecolor='dimgray', alpha=0.6)
        ax_rgb.add_feature(cfeature.STATES, linewidth=0.3, edgecolor='gray')
        gl = ax_rgb.gridlines(draw_labels=True, linewidth=0.4, alpha=0.5, linestyle='--')
        gl.top_labels = gl.right_labels = False

        for sp, grp in nearby.drop_duplicates(['latitude','longitude','common_name']).groupby('common_name'):
            ax_rgb.scatter(grp['longitude'], grp['latitude'],
                       c='none', s=55, alpha=0.95,
                       edgecolors=SPECIES_COLORS.get(sp, 'black'), linewidths=1.5,
                       marker=species_markers.get(sp, 'o'),
                       transform=ccrs.PlateCarree(), zorder=5)

        ax_rgb.set_title('Phytoplankton Community Composition (RGB composite)\n'
                         'R = Synechococcus  \u00b7  G = Picoeukaryotes  \u00b7  B = Prochlorococcus  '
                         '(gray = turbid OWT / no data)',
                         fontsize=13, fontweight='bold')

        # ── RGB triangle legend ────────────────────────────────────────────
        triangle = np.array([
            [0.5, np.sqrt(3) / 2],  # top = Pico (Green)
            [0, 0],                 # bottom left = Syn (Red)
            [1, 0],                 # bottom right = Pro (Blue)
        ])
        res = 100
        bary_coords = np.array([
            [i / res, j / res, 1 - i / res - j / res]
            for i in range(res + 1) for j in range(res + 1 - i)
        ])
        rgb_vals  = np.clip(bary_coords[:, [1, 0, 2]], 0, 1)  # R=Syn, G=Pico, B=Pro
        xy_coords = (bary_coords[:, 0:1] * triangle[0] +
                     bary_coords[:, 1:2] * triangle[1] +
                     bary_coords[:, 2:3] * triangle[2])

        legend_ax = inset_axes(ax_rgb, width="16%", height="70%", loc='center left',
                               bbox_to_anchor=(1.03, 0.0, 1, 1),
                               bbox_transform=ax_rgb.transAxes, borderpad=0)
        legend_ax.set_aspect('equal')
        legend_ax.axis('off')
        legend_ax.scatter(xy_coords[:, 0], xy_coords[:, 1], c=rgb_vals, s=3)
        legend_ax.add_patch(mpatches.Polygon(triangle, closed=True, edgecolor='black',
                                              facecolor='none', linewidth=1))
        legend_ax.text(0.5, np.sqrt(3) / 2 + 0.07, 'Picoeukaryotes', ha='center', fontsize=8)
        legend_ax.text(-0.05, -0.05, 'Synechococcus', ha='right', va='top', fontsize=8)
        legend_ax.text(1.05, -0.05, 'Prochlorococcus', ha='left', va='top', fontsize=8)
        legend_ax.set_xlim(-0.15, 1.15)
        legend_ax.set_ylim(-0.12, 1.05)
    else:
        ax_rgb.set_title('RGB composite unavailable (missing MOANA variable)')

    fig.legend(handles=legend_handles, loc='lower center',
               ncol=max(len(legend_handles), 1), fontsize=10, framealpha=0.9,
               bbox_to_anchor=(0.5, -0.02))
    plt.tight_layout()
    save_fig(save_name)
    plt.show()
    ds.close()

# Plot for the two survey dates highlighted in the presentation
plot_moana_date('2025-01-14', df, 'Fig2_MOANA_Jan14_2025.tiff')
plot_moana_date('2025-02-22', df, 'Fig3_MOANA_Feb22_2025.tiff')


## 5. Environmental Conditions by Species

Box plots of key environmental variables at observation locations, stratified by
species (Slide 6). We include all species with n ≥ 20 observations: Fin whale,
Humpback whale, Sperm whale, Common minke whale, and Beaked whales.

In [ ]:
# ── Species with n ≥ 20 ─────────────────────────────────────────────────
ANALYSIS_SPECIES = sorted(df['common_name'].value_counts()[
    df['common_name'].value_counts() >= 20].index.tolist())
print("Analysis species (n ≥ 20):", ANALYSIS_SPECIES)

df5 = df[df['common_name'].isin(ANALYSIS_SPECIES)].copy()
print(f"Observations: {len(df5):,}")

In [ ]:
# ── Figure: Chlorophyll-a, Bathymetry, SST, Phytoplankton Carbon ─────────
# (Slide 6 – 2×2 layout)
ENV_VARS = {
    'chlor_a_median':       ('Chlorophyll-a (mg m⁻³)',       'Concentration of Chlorophyll a'),
    'depth_m':              ('Depth (m)',                     'Bathymetry'),
    'sst_median':           ('SST (°C)',                      'Sea Surface Temperature'),
    'carbon_phyto_median':  ('Phytoplankton Carbon (mg m⁻³)','Conc. of Phytoplankton Carbon'),
}

species_order = ANALYSIS_SPECIES
colors = [SPECIES_COLORS.get(s, 'grey') for s in species_order]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for ax, (var, (ylabel, title)) in zip(axes, ENV_VARS.items()):
    data_by_sp = [df5[df5['common_name'] == sp][var].dropna() for sp in species_order]
    bp = ax.boxplot(data_by_sp,
                    labels=[s.replace(' whale','').replace('Common ','') for s in species_order],
                    patch_artist=True, showmeans=True, showfliers=True,
                    meanprops=dict(marker='D', markerfacecolor='red', markersize=7))
    for patch, col in zip(bp['boxes'], colors):
        patch.set_facecolor(col); patch.set_alpha(0.65)

    if var == 'depth_m':
        ax.invert_yaxis()
    ax.set_ylabel(ylabel, fontsize=11, fontweight='bold')
    ax.set_title(f'{title} Distribution by Species
'
                 f'(Box: IQR, Red Diamond: Mean, Line: Median)',
                 fontsize=12, fontweight='bold')
    ax.tick_params(axis='x', rotation=30)
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
save_fig('Fig4_BoxPlots_EnvVars_by_Species.tiff')
plt.show()

### 5b. MOANA Prokaryote Abundance by Species (Slide 9)

Box plots for the three MOANA phytoplankton community variables, shown
separately for each species.

In [ ]:
# ── Figure: Synechococcus / Prochlorococcus / Picoeukaryotes ──────────────
MOANA_VARS = {
    'synechococcus_median':  ('Synechococcus (cells mL⁻¹)',  'Synechococcus'),
    'prochlorococcus_median':('Prochlorococcus (cells mL⁻¹)','Prochlorococcus'),
    'picoeukaryotes_median': ('Picoeukaryotes (cells mL⁻¹)', 'Picoeukaryotes'),
}

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
fig.suptitle('Prokaryote Abundance Distribution by Species
(Box: IQR, Red Diamond: Mean, Line: Median)',
             fontsize=14, fontweight='bold')

for ax, (var, (ylabel, title)) in zip(axes, MOANA_VARS.items()):
    data_by_sp = [df5[df5['common_name'] == sp][var].dropna() for sp in species_order]
    bp = ax.boxplot(data_by_sp,
                    labels=[s.replace(' whale','').replace('Common ','') for s in species_order],
                    patch_artist=True, showmeans=True,
                    meanprops=dict(marker='D', markerfacecolor='red', markersize=7))
    for patch, col in zip(bp['boxes'], colors):
        patch.set_facecolor(col); patch.set_alpha(0.65)

    ax.set_ylabel(ylabel, fontsize=11, fontweight='bold')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.ticklabel_format(axis='y', style='scientific', scilimits=(0,0))
    ax.tick_params(axis='x', rotation=30)
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
save_fig('Fig5_MOANA_Prokaryote_Abundance_by_Species.tiff')
plt.show()

## 6. Environmental Niche Analysis

The environmental niche analysis quantifies how differently each species uses
its habitat by measuring the overlap in the distribution of each environmental
variable across species pairs. A niche overlap of 0 = no overlap;
1 = complete overlap (Pianka's index approximated via histogram bins).

**Species included:** all with n ≥ 20 (fin, humpback, sperm, minke, beaked whales).


### 6a. All-variable summary statistics & Kruskal-Wallis tests

In [ ]:
# ── Environmental variable list for niche analysis ────────────────────────
ENV_NICHE = {
    'chlor_a_median':        'Chlorophyll-a',
    'chlor_a_std':           'Chlorophyll Variability',
    'synechococcus_median':  'Synechococcus',
    'prochlorococcus_median':'Prochlorococcus',
    'picoeukaryotes_median': 'Picoeukaryotes',
    'carbon_phyto_median':   'Particulate Organic Carbon',
    'sst_median':            'Sea Surface Temperature',
    'depth_m':               'Water Depth',
    'latitude':              'Latitude',
    'ssha':                  'SSH Anomaly',
}

df6 = df5.copy()

# ── Summary statistics per species ───────────────────────────────────────
print("="*70)
print("ENVIRONMENTAL CONDITIONS BY SPECIES")
print("="*70)
summary_rows = []
for sp in ANALYSIS_SPECIES:
    spd = df6[df6['common_name'] == sp]
    row = {'Species': sp, 'N': len(spd)}
    for var, label in ENV_NICHE.items():
        row[f'{label}_mean'] = spd[var].mean() if var in spd else np.nan
        row[f'{label}_std']  = spd[var].std()  if var in spd else np.nan
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
print(summary_df[['Species','N']].to_string(index=False))

# ── Kruskal-Wallis tests ─────────────────────────────────────────────────
print("\n" + "="*70)
print("KRUSKAL-WALLIS TESTS (significant differences across species?)")
print("="*70)
kw_results = []
for var, label in ENV_NICHE.items():
    if var not in df6.columns:
        continue
    groups = [df6[df6['common_name'] == sp][var].dropna().values
              for sp in ANALYSIS_SPECIES]
    groups = [g for g in groups if len(g) >= 3]
    if len(groups) < 2:
        continue
    h, p = kruskal(*groups)
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    kw_results.append({'Variable': label, 'H': round(h,3), 'p': round(p,4), 'Sig': sig})
    print(f"  {label:35s}  H={h:.3f}  p={p:.4f}  {sig}")

kw_df = pd.DataFrame(kw_results)
kw_df.to_csv('data/KruskalWallis_Results.csv', index=False)

### 6b. Post-hoc Pairwise Comparisons (Mann-Whitney U)

We run pairwise Mann-Whitney U tests between all species pairs for each variable,
with Bonferroni correction for the number of comparisons.

In [ ]:
# ── Post-hoc pairwise Mann-Whitney U ─────────────────────────────────────
n_pairs          = len(list(combinations(ANALYSIS_SPECIES, 2)))
alpha_bonferroni = 0.05 / n_pairs
print(f"Bonferroni-corrected α = 0.05 / {n_pairs} = {alpha_bonferroni:.4f}\n")

pairwise_rows = []
for var, label in ENV_NICHE.items():
    if var not in df6.columns:
        continue
    print(f"\n{label}:")
    print("-"*60)
    for sp1, sp2 in combinations(ANALYSIS_SPECIES, 2):
        d1 = df6[df6['common_name'] == sp1][var].dropna().values
        d2 = df6[df6['common_name'] == sp2][var].dropna().values
        if len(d1) < 3 or len(d2) < 3:
            continue
        u, p = mannwhitneyu(d1, d2, alternative='two-sided')
        sig  = ("***" if p < alpha_bonferroni else "**" if p < 0.01
                else "*" if p < 0.05 else "ns")
        short = lambda s: s.replace(' whale','').replace('Common ','')
        print(f"  {short(sp1):14s} vs {short(sp2):14s}: U={u:9.1f}  p={p:.4f}  {sig}")
        pairwise_rows.append({'Variable': label, 'Species1': sp1, 'Species2': sp2,
                              'U': u, 'p': p, 'Sig': sig})

pd.DataFrame(pairwise_rows).to_csv('data/PairwiseMannWhitney_Results.csv', index=False)
print("\nSaved pairwise results to data/PairwiseMannWhitney_Results.csv")

### 6c. Niche Overlap Calculation

We compute pairwise niche overlap for each environmental variable using Pianka's
index approximated via histogram bins (50 bins over the combined range). Values
range from 0 (no overlap) to 1 (identical distributions).

In [ ]:
def pianka_overlap(x, y, n_bins=50):
    """Approximate Pianka niche overlap from two 1-D arrays of values."""
    if len(x) < 3 or len(y) < 3:
        return np.nan
    xmin = min(x.min(), y.min())
    xmax = max(x.max(), y.max())
    if xmax == xmin:
        return np.nan
    bins   = np.linspace(xmin, xmax, n_bins + 1)
    px, _  = np.histogram(x, bins=bins, density=True)
    py, _  = np.histogram(y, bins=bins, density=True)
    px     = px / px.sum() if px.sum() > 0 else px
    py     = py / py.sum() if py.sum() > 0 else py
    num    = np.sum(px * py)
    denom  = np.sqrt(np.sum(px**2) * np.sum(py**2))
    return float(num / denom) if denom > 0 else np.nan

# ── Compute overlaps for all variable × species-pair combinations ─────────
overlap_rows = []
for var, label in ENV_NICHE.items():
    if var not in df6.columns:
        continue
    for i, sp1 in enumerate(ANALYSIS_SPECIES):
        for sp2 in ANALYSIS_SPECIES[i+1:]:
            x = df6[df6['common_name'] == sp1][var].dropna().values
            y = df6[df6['common_name'] == sp2][var].dropna().values
            ov = pianka_overlap(x, y)
            cat = ('High' if not np.isnan(ov) and ov >= 0.7 else
                   'Moderate' if not np.isnan(ov) and ov >= 0.4 else
                   'Low' if not np.isnan(ov) and ov >= 0.2 else
                   'Very Low' if not np.isnan(ov) else 'No Data')
            sp1s = sp1.replace(' whale','').replace('Common ','')
            sp2s = sp2.replace(' whale','').replace('Common ','')
            overlap_rows.append({'Variable': label, 'Species_1': sp1s, 'Species_2': sp2s,
                                 'Pair': f'{sp1s}-{sp2s}', 'Overlap': ov, 'Category': cat})

overlap_df = pd.DataFrame(overlap_rows)
overlap_df.to_csv('data/Niche_Overlap_Summary.csv', index=False)
print("Niche overlap summary:")
print(overlap_df.pivot_table(index='Pair', columns='Variable', values='Overlap').round(3))

### 6d. Niche Overlap Dot Plot – Phytoplankton & Carbon (Slide 12)

Each point is the niche overlap score for one phytoplankton variable × species
pair. Horizontal bands indicate overlap categories.

In [ ]:
# ── Dot plot: phytoplankton + carbon niche overlap ───────────────────────
BIO_VARS  = ['Synechococcus','Prochlorococcus','Picoeukaryotes','Particulate Organic Carbon']
BIO_COLORS = {'Synechococcus':          '#e74c3c',
               'Prochlorococcus':        '#3498db',
               'Picoeukaryotes':         '#2ecc71',
               'Particulate Organic Carbon': '#f39c12'}

pairs_short = sorted(overlap_df['Pair'].unique())
x_pos = np.arange(len(pairs_short))

fig, ax = plt.subplots(figsize=(14, 8))

# Reference zones
for lo, hi, col in [(0, 0.2,'red'),(0.2,0.4,'orange'),(0.4,0.7,'yellow'),(0.7,1.0,'green')]:
    ax.axhspan(lo, hi, alpha=0.08, color=col, zorder=0)
for th in [0.2, 0.4, 0.7]:
    ax.axhline(th, color='grey', lw=0.8, ls='--', alpha=0.5)

# Category labels
for y, lbl, col in [(0.1,'Very Low\n(<0.2)','red'),(0.3,'Low\n(0.2–0.4)','orange'),
                     (0.55,'Moderate\n(0.4–0.7)','#d4ac0d'),(0.85,'High\n(>0.7)','green')]:
    ax.text(-0.6, y, lbl, ha='center', va='center', fontsize=9, color=col,
            fontweight='bold', bbox=dict(boxstyle='round', fc='white', ec=col, alpha=0.9))

for var_label, col in BIO_COLORS.items():
    sub   = overlap_df[overlap_df['Variable'] == var_label].set_index('Pair')
    y_val = [sub['Overlap'].get(p, np.nan) for p in pairs_short]
    ax.plot(x_pos, y_val, '-', color=col, alpha=0.35, lw=2, zorder=1)
    ax.scatter(x_pos, y_val, s=180, color=col, alpha=0.85,
               edgecolors='black', lw=1.5, label=var_label, zorder=2)

ax.set_xticks(x_pos)
ax.set_xticklabels([p.replace('-','
vs
') for p in pairs_short], fontsize=10)
ax.set_xlabel('Species Pair', fontweight='bold', fontsize=12)
ax.set_ylabel('Niche Overlap (0 = No overlap, 1 = Complete overlap)', fontweight='bold', fontsize=12)
ax.set_title('Phytoplankton & Carbon Niche Overlap: Species Pairs Comparison',
             fontweight='bold', fontsize=14)
ax.set_ylim(-0.05, 1.05)
ax.legend(fontsize=10, loc='upper right')
ax.grid(True, alpha=0.2, axis='y')

plt.tight_layout()
save_fig('Fig6_NicheOverlap_BioVars_DotPlot.tiff')
plt.show()

### 6e. Biological vs Physical Variable Niche Overlap Comparison (Slide 13)

Side-by-side comparison of niche overlap calculated from biological variables
(phytoplankton + carbon) and physical/geographic variables
(depth, latitude, chlorophyll-a, SST).

In [ ]:
# ── Biological vs Physical niche overlap comparison ──────────────────────
PHYS_VARS   = {'Water Depth': '#34495e', 'Latitude': '#e67e22',
               'Chlorophyll-a': '#8e44ad', 'Sea Surface Temperature': '#e74c3c'}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8), sharey=True)
fig.suptitle('Niche Overlap: Biological vs Physical Variables',
             fontsize=14, fontweight='bold')

for ax, var_dict, panel_title in [
    (ax1, BIO_COLORS, 'A) Biological Variables\n(Phytoplankton & Carbon)'),
    (ax2, PHYS_VARS,  'B) Physical/Geographic Variables\n(Depth, Location, Productivity, Temperature)')]:

    for lo, hi, col in [(0,0.2,'red'),(0.2,0.4,'orange'),(0.4,0.7,'yellow'),(0.7,1.0,'green')]:
        ax.axhspan(lo, hi, alpha=0.07, color=col, zorder=0)
    for th in [0.2, 0.4, 0.7]:
        ax.axhline(th, color='grey', lw=0.8, ls='--', alpha=0.4)

    for var_label, col in var_dict.items():
        sub   = overlap_df[overlap_df['Variable'] == var_label].set_index('Pair')
        y_val = [sub['Overlap'].get(p, np.nan) for p in pairs_short]
        ax.plot(x_pos, y_val, '-', color=col, alpha=0.35, lw=2, zorder=1)
        ax.scatter(x_pos, y_val, s=180, color=col, alpha=0.85,
                   edgecolors='black', lw=1.5, label=var_label, zorder=2)

    ax.set_xticks(x_pos)
    ax.set_xticklabels([p.replace('-','-\n') for p in pairs_short],
                       fontsize=10, rotation=-20, ha='left')
    ax.set_xlabel('Species Pair', fontweight='bold', fontsize=12)
    ax.set_title(panel_title, fontweight='bold', fontsize=13)
    ax.set_ylim(-0.05, 1.05)
    ax.legend(fontsize=10, loc='upper right')
    ax.grid(True, alpha=0.2, axis='y')

ax1.set_ylabel('Niche Overlap (0 = No overlap, 1 = Complete overlap)',
               fontweight='bold', fontsize=12)

plt.tight_layout()
save_fig('Fig7_NicheOverlap_Bio_vs_Physical.tiff')
plt.show()

## 7. Multivariate Analysis: Principal Component Analysis (PCA)

PCA reduces the high-dimensional environmental space to a small number of orthogonal
axes, letting us visualise how species separate (or overlap) in that space and —
crucially for an RSE audience — which satellite variables drive the separation.

**What to look for in the output:**
- **Variance explained by PC1 + PC2**: if ≥ ~50–60% combined you can tell a coherent
  story; lower values indicate genuinely complex environmental space (worth stating).
- **Loading arrows**: long arrows pointing in the same direction as a species cluster
  identify the key habitat-differentiating satellite products. E.g. if Prochlorococcus
  points toward the sperm whale cluster and Picoeukaryotes toward the baleen whale
  cluster, that corroborates the trophic pathway story.
- **Species clusters**: do baleen whales (fin, humpback, minke) group together and
  separate from deep-diving odontocetes (sperm, beaked)? Consistency with the niche
  overlap results strengthens both analyses.

The analytical arc is: **box plots → Kruskal-Wallis/Mann-Whitney (statistical
differences) → niche overlap (how different, and in which variables) → PCA (which
environmental axes explain the differences)**. Each step adds a layer the previous
one cannot provide on its own.

In [ ]:
# ── Section 7: PCA (only multivariate analysis retained) ────────────────
# Hierarchical clustering was removed: the dendrogram restates niche
# overlap results without adding new information, and is underpowered
# for n=5 species. PCA adds interpretive value by identifying WHICH
# environmental axes (satellite products) drive inter-species differences.

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ── Prepare data: drop rows with any NaN in analysis variables ────────────
pca_vars = [v for v in ENV_NICHE if v in df6.columns and v not in ('latitude','ssha')]
df_pca   = df6[pca_vars + ['common_name']].dropna()
print(f"PCA dataset: {len(df_pca):,} observations × {len(pca_vars)} variables")
print("Observations per species:")
print(df_pca['common_name'].value_counts())

X       = df_pca[pca_vars].values
species = df_pca['common_name'].values

scaler  = StandardScaler()
X_sc    = scaler.fit_transform(X)

pca   = PCA(n_components=len(pca_vars))
X_pca = pca.fit_transform(X_sc)

pca_df = pd.DataFrame(X_pca[:, :3], columns=['PC1','PC2','PC3'])
pca_df['Species'] = species

print("\nVariance explained:")
for i, v in enumerate(pca.explained_variance_ratio_[:4]):
    print(f"  PC{i+1}: {v*100:.1f}%")
print(f"  Cumulative PC1–3: {pca.explained_variance_ratio_[:3].sum()*100:.1f}%")

In [ ]:
# ── PCA biplot: PC1 vs PC2 and PC2 vs PC3 ────────────────────────────────
var_labels = [ENV_NICHE.get(v, v) for v in pca_vars]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('PCA: Environmental Niche Space', fontsize=14, fontweight='bold')

for ax, (pc_x, pc_y), ax_title in [
    (ax1, (0, 1), 'PC1 vs PC2'),
    (ax2, (1, 2), 'PC2 vs PC3')]:

    for sp in ANALYSIS_SPECIES:
        mask = pca_df['Species'] == sp
        ax.scatter(pca_df[mask][f'PC{pc_x+1}'], pca_df[mask][f'PC{pc_y+1}'],
                   c=SPECIES_COLORS.get(sp,'grey'), alpha=0.5, s=30,
                   edgecolors='black', lw=0.3,
                   label=sp.replace(' whale','').replace('Common ',''))

    # Loading vectors (only on first panel)
    if pc_x == 0:
        loadings = pca.components_[[pc_x, pc_y]].T * np.sqrt(pca.explained_variance_[[pc_x, pc_y]])
        for i, lbl in enumerate(var_labels):
            ax.annotate('', xy=(loadings[i,0]*4, loadings[i,1]*4), xytext=(0,0),
                        arrowprops=dict(arrowstyle='->', color='darkred', lw=1.5))
            ax.text(loadings[i,0]*4.4, loadings[i,1]*4.4, lbl,
                    fontsize=8, color='darkred', fontweight='bold')

    ax.axhline(0, color='k', lw=0.5, alpha=0.4)
    ax.axvline(0, color='k', lw=0.5, alpha=0.4)
    ax.set_xlabel(f'PC{pc_x+1} ({pca.explained_variance_ratio_[pc_x]*100:.1f}%)',
                  fontweight='bold', fontsize=12)
    ax.set_ylabel(f'PC{pc_y+1} ({pca.explained_variance_ratio_[pc_y]*100:.1f}%)',
                  fontweight='bold', fontsize=12)
    ax.set_title(ax_title, fontsize=13)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
save_fig('Fig8_PCA_Environmental_Niche.tiff')
plt.show()